# Reconocimiento e Identificación de Placas Vehiculares — v6.1 (parches)
### Módulo 3 · Visión por Computadora · Diplomado RNA y Deep Learning
---
## Mejora principal: buscar el texto MÁS GRANDE dentro de la placa

### Diagnóstico v5
En v5 todos los textos de la placa competían igual en el OCR:
```
placa1 fallback → lee 'AN ADMS MIE LRY-59-77'
              ↑ código de barras + logo + número → todo mezclado
placa3         → lee 'EDOMEX' en lugar de 'NLU-96-03'
              ↑ la región detectada cae sobre el logo, no los números
```

### Solución: `image_to_data()` con filtro por altura de caracteres

Tesseract tiene dos modos de salida:
- `image_to_string()` → devuelve todo el texto (no distingue tamaños)
- `image_to_data()`   → devuelve cada palabra con su posición y dimensiones

En TODAS las placas vehiculares, el número de placa es el texto
con caracteres más grandes. El logo del estado, el código de barras
y el texto legal son mucho más pequeños.

```
placa1: 'LRV-59-77' → altura ~60px ← queremos esto
        'ESTADO DE MEXICO' → altura ~15px  
        código de barras  → altura ~8px   → filtrados
```

Filtramos palabras donde `height >= 25% de la altura de la imagen`.

### Segunda mejora: detección de placa CO por color

El Haar cascade no detecta bien las placas amarillas colombianas
(fue entrenado principalmente con placas claras europeas).
Solución alternativa: buscar directamente regiones amarillas rectangulares
con el aspect ratio correcto (2.0–3.5 para placas CO).

### Propuesta de arquitectura futura (trabajo futuro)
Para escenas de calle complejas (placa5, 6, 7) la solución robusta sería
un **pipeline en cascada**:
1. Detectar vehículos (YOLO, SSD, o cascade de autos)
2. Dentro de cada vehículo, buscar la placa
3. OCR en la región de placa confirmada

Esto eliminaría los falsos positivos de árboles, letreros y el anuncio
de Coppel que aparecen en placa6 y placa7.

## Historial de versiones

| v | Mejora |
|---|--------|
| 1 | Estructura base |
| 2 | `cargar_imagenes()`, detección CO, rutas multiplataforma |
| 3 | Filtro aspect ratio, morfología, zona de números |
| 4 | Filtro contraste (árboles), scoring por formato (guiones) |
| 5 | Auto-inversión, thresholds fijos (técnica de clase) |
| **6** | **`image_to_data()` + altura chars, detección CO por color** |


---
## Notas de Instalación — Tesseract OCR
### Windows 11
```
1. https://github.com/UB-Mannheim/tesseract/wiki → tesseract-ocr-w64-setup-*.exe
2. Instalar. Ruta: C:\Program Files\Tesseract-OCR\tesseract.exe
3. Reiniciar kernel.
```
### Ubuntu
```bash
sudo apt update && sudo apt install tesseract-ocr libtesseract-dev
```
### Google Colab
```python
!sudo apt install tesseract-ocr -q && pip install pytesseract -q
from google.colab import drive; drive.mount('/content/drive')
```


In [1]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports y configuración global
# ═══════════════════════════════════════════════════════════════

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import platform
import pytesseract
from pytesseract import Output

if platform.system() == 'Windows':
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

RUTA_PLACAS     = os.path.join('..', 'Material', 'Plates')
EXTENSIONES_IMG = ('.jpg', '.jpeg', '.png', '.bmp')

# Filtros de detección
AR_MIN, AR_MAX   = 1.5, 7.0
BRILLO_MIN      = 35             # PARCHE 2: bajado de 100→35. Árboles≈15-25, placas oscuras≈35-80
KERNEL_MORPH     = np.ones((2, 2), np.uint8)

# Thresholds fijos (técnica de clase)
THRESHOLDS_FIJOS = [100, 110, 130, 160]

# Filtro de altura mínima para text boxes (image_to_data)
# Proporción mínima respecto a la altura de la imagen
ALTURA_MIN_RATIO = 0.25   # 25% de la altura → ignora texto pequeño

OCR_CFG = {
    'MX' : '--psm 7 --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-',
    'EU' : '--psm 7 --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
    'CO' : '--psm 7 --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-',
    'GEN': '--psm 6 --oem 3',
}

GROUND_TRUTH = {
    'placa1.jpg': ('LRV5977', 'MX'),
    'placa2.jpg': ('5649JSN', 'EU'),
    'placa3.jpg': ('NLU9603', 'MX'),
    'placa4.jpg': ('NAA7741', 'MX'),
    'placa5.jpg': ('BNR249',  'CO'),
    'placa6.jpg': ('NNR7087', 'MX'),
    'placa7.jpg': ('470MJV',  'MX'),
}

RANGO_CHARS = {'MX':(5,8), 'EU':(6,8), 'CO':(5,7), 'GEN':(4,10)}

print(f'Sistema   : {platform.system()}')
print(f'Ruta      : {os.path.abspath(RUTA_PLACAS)}')
print(f'Tesseract : {pytesseract.get_tesseract_version()}')
print('Imports listos ✅')


Sistema   : Windows
Ruta      : f:\Proyectos\Diplomado-RNA\Modulo-3\Material\Plates
Tesseract : 5.5.0.20241111
Imports listos ✅


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Haar Cascade
# ═══════════════════════════════════════════════════════════════
plate_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_russian_plate_number.xml'
)
print('Haar Cascade cargado ✅')


Haar Cascade cargado ✅


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3a — cargar_imagenes()
# ═══════════════════════════════════════════════════════════════

def cargar_imagenes(ruta_carpeta, prefijos=None, ordenar=True):
    """Escanea carpeta. Compatible con datasets de cualquier tamaño."""
    if not os.path.isdir(ruta_carpeta):
        print(f'Carpeta no encontrada: {os.path.abspath(ruta_carpeta)}')
        return []
    archivos = [
        os.path.join(ruta_carpeta, n)
        for n in os.listdir(ruta_carpeta)
        if n.lower().endswith(EXTENSIONES_IMG)
        and (not prefijos or n.lower().startswith(prefijos))
    ]
    return sorted(archivos) if ordenar else archivos

imagenes = cargar_imagenes(RUTA_PLACAS, prefijos=('placa',))
print(f'Imágenes: {len(imagenes)}')
for r in imagenes:
    n = os.path.basename(r)
    gt, tp = GROUND_TRUTH.get(n, ('?','?'))
    print(f'  {n:<12} [{gt}] {tp}')


Imágenes: 7
  placa1.jpg   [LRV5977] MX
  placa2.jpg   [5649JSN] EU
  placa3.jpg   [NLU9603] MX
  placa4.jpg   [NAA7741] MX
  placa5.jpg   [BNR249] CO
  placa6.jpg   [NNR7087] MX
  placa7.jpg   [470MJV] MX


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3b — identificar_tipo_placa() + filtrar_detecciones()
# ═══════════════════════════════════════════════════════════════

def identificar_tipo_placa(img_rgb):
    """
    Detecta EU (franja azul izq), CO (fondo amarillo) o MX (default).
    """
    h, w = img_rgb.shape[:2]
    hsv  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    franja   = hsv[:, :int(w*0.15)]
    mask_eu  = cv2.inRange(franja, np.array([100,100,80]), np.array([135,255,255]))
    score_eu = cv2.countNonZero(mask_eu) / max(franja.shape[0]*franja.shape[1], 1) * 100
    mask_co  = cv2.inRange(hsv, np.array([20,150,150]), np.array([35,255,255]))
    score_co = cv2.countNonZero(mask_co) / max(h*w, 1) * 100
    info = {'eu': round(score_eu,1), 'co': round(score_co,1)}
    if score_eu > 20: return 'EU', info
    if score_co > 12: return 'CO', info
    return 'MX', info


def filtrar_detecciones(dets, img_rgb):
    """Filtra por aspect ratio (1.5-7.0) y contraste stddev > 100."""
    validas, descartadas = [], []
    for det in dets:
        x, y, w, h = det
        ar  = round(w/h, 2) if h else 0
        rec = img_rgb[y:y+h, x:x+w]
        std = round(float(np.std(cv2.cvtColor(rec, cv2.COLOR_RGB2GRAY))), 1)
        if not (AR_MIN <= ar <= AR_MAX):
            descartadas.append(f'AR={ar}')
        elif std < BRILLO_MIN:
            descartadas.append(f'contraste={std}<{BRILLO_MIN}')
        else:
            validas.append(det)
    return validas, descartadas

print('identificar_tipo_placa() lista ✅')
print('filtrar_detecciones() lista ✅')


identificar_tipo_placa() lista ✅
filtrar_detecciones() lista ✅


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3c — buscar_placa_amarilla()  ← NUEVA en v6
# ═══════════════════════════════════════════════════════════════
#
# PROBLEMA: el Haar cascade fue entrenado con placas claras/blancas.
# Las placas colombianas (amarillas) no siempre las detecta bien.
#
# SOLUCIÓN ALTERNATIVA: buscar directamente regiones amarillas
# que tengan el aspect ratio de una placa (2.0–3.5 para CO).
#
# Método:
#   1. Crear máscara de color amarillo en HSV
#   2. Encontrar contornos en la máscara
#   3. Para cada contorno, calcular bounding box y aspect ratio
#   4. Si AR está en rango de placa CO → candidato
#   5. Devolver el candidato más grande

def buscar_placa_amarilla(img_rgb, ar_min=2.0, ar_max=3.8, area_min=500):
    """
    Busca regiones amarillas con aspect ratio de placa colombiana.

    Args:
        img_rgb  : imagen original en RGB
        ar_min   : aspect ratio mínimo (w/h)
        ar_max   : aspect ratio máximo
        area_min : área mínima en píxeles (filtra ruido pequeño)

    Returns:
        recorte  : imagen recortada o None si no encuentra
        bbox     : (x,y,w,h) o None
    """
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

    # Rango del amarillo colombiano en HSV
    mask = cv2.inRange(img_hsv,
                       np.array([18, 120, 120]),
                       np.array([38, 255, 255]))

    # Operación morfológica para cerrar huecos en la máscara
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidatos = []
    for cnt in contornos:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        ar   = w / h if h > 0 else 0
        if area >= area_min and ar_min <= ar <= ar_max:
            candidatos.append((area, x, y, w, h))

    if not candidatos:
        return None, None

    # Tomar el candidato de mayor área
    candidatos.sort(reverse=True)
    _, x, y, w, h = candidatos[0]

    recorte = img_rgb[y:y+h, x:x+w]
    return recorte, (x, y, w, h)

print('buscar_placa_amarilla() lista ✅')


buscar_placa_amarilla() lista ✅


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3d — detectar_placas()  + score_ocr()
# ═══════════════════════════════════════════════════════════════

def score_ocr_resultado(texto, tipo):
    """Puntúa por formato: longitud en rango + guiones + mezcla chars."""
    if not texto: return -99
    chars = texto.replace('-','')
    n_ch  = len(chars)
    n_gui = texto.count('-')
    mix   = any(c.isalpha() for c in chars) and any(c.isdigit() for c in chars)
    rmin, rmax = RANGO_CHARS.get(tipo, (4,10))
    s = 0
    s += 20 if rmin <= n_ch <= rmax else -5*(max(0,n_ch-rmax)+max(0,rmin-n_ch))
    if tipo in ('MX','CO'): s += 15*min(n_gui,2)
    elif tipo=='EU':        s -= 5*n_gui
    if mix: s += 10
    s += n_ch
    return s


def auto_invertir(img_bin):
    """Si la imagen es predominantemente oscura (invertida), aplica bitwise_not."""
    if np.mean(img_bin) < 127:
        return cv2.bitwise_not(img_bin), True
    return img_bin, False


def detectar_placas(ruta_imagen):
    """
    4 intentos con Haar + filtros AR/contraste.
    Para imágenes donde Haar falla → fallback imagen completa.
    """
    img_bgr = cv2.imread(ruta_imagen)
    if img_bgr is None:
        print(f'  No se pudo cargar: {ruta_imagen}')
        return None, [], [], -1

    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    intentos = [
        (1.10, 5, (60,20), 'estandar'),
        (1.05, 3, (40,15), 'permisivo'),
        (1.03, 1, (30,10), 'ultra-permisivo'),
        (1.03, 1, (20, 8), 'escena-calle'),
    ]

    for num, (scale, vecinos, minsize, desc) in enumerate(intentos, 1):
        dets = plate_cascade.detectMultiScale(
            img_gris, scaleFactor=scale, minNeighbors=vecinos, minSize=minsize)
        if len(dets) == 0: continue
        validas, desc_list = filtrar_detecciones(dets, img_rgb)
        if desc_list:
            print(f'  Intento {num}: {len(desc_list)} rechazada(s): {desc_list}')
        if not validas: continue
        dets_s   = sorted(validas, key=lambda r: r[2]*r[3], reverse=True)
        recortes = [img_rgb[y:y+h, x:x+w] for (x,y,w,h) in dets_s]
        print(f'  Intento {num} ({desc}): {len(validas)} válida(s)')
        return img_rgb, recortes, dets_s, num

    print('  Sin detecciones → fallback imagen completa')
    hf, wf = img_rgb.shape[:2]
    return img_rgb, [img_rgb], [(0,0,wf,hf)], 0

print('detectar_placas() lista ✅')
print('score_ocr_resultado() lista ✅')
print('auto_invertir() lista ✅')


detectar_placas() lista ✅
score_ocr_resultado() lista ✅
auto_invertir() lista ✅


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3e — ocr_por_altura()  ← NUEVA en v6
# ═══════════════════════════════════════════════════════════════
#
# IDEA: en todas las placas vehiculares, el número de placa tiene
# los caracteres MÁS GRANDES. El logo del estado, el código de barras
# y el texto legal son mucho más pequeños.
#
# Usamos pytesseract.image_to_data() que devuelve:
#   - text   : texto de cada palabra detectada
#   - left, top, width, height : posición y tamaño del bounding box
#   - conf   : confianza (0-100)
#
# Filtramos por:
#   height >= ALTURA_MIN_RATIO * altura_imagen
#   conf >= 0  (incluimos todo, el scoring ya filtra)
#
# Devolvemos también los bounding boxes para visualización.

def ocr_por_altura(img_bin, tipo, altura_min_ratio=ALTURA_MIN_RATIO):
    """
    OCR con filtrado por altura de caracteres.
    Selecciona solo el texto con caracteres grandes (número de placa).

    Args:
        img_bin          : imagen binarizada y orientada correctamente
        tipo             : 'MX', 'EU', 'CO' para config OCR
        altura_min_ratio : proporción mínima de altura (0.25 = 25%)

    Returns:
        texto  : str → palabras de texto grande unidas
        boxes  : lista de (x,y,w,h) de las palabras seleccionadas
        todos  : lista de (x,y,w,h,text,conf) de TODO el texto
    """
    config_ocr    = OCR_CFG.get(tipo, OCR_CFG['GEN'])
    h_img, w_img  = img_bin.shape[:2]
    altura_min    = h_img * altura_min_ratio

    data = pytesseract.image_to_data(
        img_bin, config=config_ocr, output_type=Output.DICT
    )

    palabras_grandes = []
    boxes            = []
    todos            = []

    for i in range(len(data['text'])):
        txt  = data['text'][i].strip()
        conf = int(data['conf'][i])
        bh   = data['height'][i]
        bx   = data['left'][i]
        by   = data['top'][i]
        bw   = data['width'][i]

        if txt:  # registrar todo lo que Tesseract detectó
            todos.append((bx, by, bw, bh, txt, conf))

        # Filtrar: altura suficiente para ser número de placa
        if txt and bh >= altura_min:
            limpio = ''.join(c for c in txt.upper() if c.isalnum() or c == '-')
            if limpio:
                palabras_grandes.append(limpio)
                boxes.append((bx, by, bw, bh))

    texto = '-'.join(palabras_grandes) if palabras_grandes else ''
    return texto, boxes, todos


# Demo de la función
print('ocr_por_altura() lista ✅')
print(f'Altura mínima de filtro: {ALTURA_MIN_RATIO*100:.0f}% de la altura de la imagen')
print('Usa pytesseract.image_to_data() → devuelve bounding boxes de cada palabra')


ocr_por_altura() lista ✅
Altura mínima de filtro: 25% de la altura de la imagen
Usa pytesseract.image_to_data() → devuelve bounding boxes de cada palabra


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3f — extraer_texto()  ← PARCHE 3: PSM adaptativo + fallback fix
# ═══════════════════════════════════════════════════════════════
#
# BUG v6: en fallback (imagen completa) se usaba PSM 7 = 'una sola línea'.
# Con la imagen completa hay muchas líneas → Tesseract devolvía basura o vacío.
# FIX:
#   - Si es fallback → usar PSM 6 (bloque de texto) en lugar de PSM 7
#   - Si es fallback → NO aplicar zone crop (el 45-90% de la imagen completa
#     recorta el auto, no la placa)

def extraer_texto(img_placa_rgb, tipo='MX', escala=2.0, es_fallback=False):
    """
    Pipeline completo con PSM adaptativo según si es fallback o no.

    Args:
        img_placa_rgb : imagen del ROI
        tipo          : 'MX', 'EU', 'CO'
        escala        : factor de escalado
        es_fallback   : True si se usa imagen completa (sin detección Haar)

    Returns:
        texto_final, detalle, scores, mejor_var, etapas
    """
    # PSM adaptativo: PSM 7 para recortes de placa, PSM 6 para imagen completa
    if es_fallback:
        config_base = '--oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-'
        config_ocr  = '--psm 6 ' + config_base   # PSM 6 = bloque de texto
    else:
        config_ocr = OCR_CFG.get(tipo, OCR_CFG['GEN'])  # PSM 7 = una línea

    # ── Crop por tipo (solo si NO es fallback) ────────────────
    h_img, w_img = img_placa_rgb.shape[:2]
    if not es_fallback:
        if tipo == 'MX':
            zona = img_placa_rgb[int(h_img*0.45): int(h_img*0.90), :]
        elif tipo == 'EU':
            zona = img_placa_rgb[:, int(w_img*0.13):]
        else:
            zona = img_placa_rgb
        img_base = zona if (zona.shape[0]>10 and zona.shape[1]>10) else img_placa_rgb
    else:
        img_base = img_placa_rgb   # fallback: imagen completa sin crop

    # ── Preprocesamiento ───────────────────────────────────────
    h, w    = img_base.shape[:2]
    img_esc = cv2.resize(img_base, (int(w*escala), int(h*escala)),
                         interpolation=cv2.INTER_CUBIC)
    g       = cv2.cvtColor(img_esc, cv2.COLOR_RGB2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    g_cla   = clahe.apply(g)
    g_med   = cv2.medianBlur(g_cla, 3)

    # ── Binarizaciones ────────────────────────────────────────
    _, b_otsu  = cv2.threshold(g_med, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    b_adapt    = cv2.adaptiveThreshold(g_med, 255,
                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    b_morph    = cv2.morphologyEx(b_otsu, cv2.MORPH_CLOSE, KERNEL_MORPH)
    bins = {'otsu': b_otsu, 'adapt': b_adapt, 'morph': b_morph}
    for t in THRESHOLDS_FIJOS:
        _, bt = cv2.threshold(g_med, t, 255, cv2.THRESH_BINARY)
        bins[f'fijo_{t}']     = bt
        bins[f'fijo_{t}_inv'] = cv2.bitwise_not(bt)

    # ── OCR + auto-inversión + extracción de patrón ───────────
    detalle = {}
    scores  = {}
    for nombre, img_bin in bins.items():
        img_corr, _ = auto_invertir(img_bin)
        raw_text    = pytesseract.image_to_string(img_corr, config=config_ocr)
        raw_limpio  = ''.join(c for c in raw_text.upper() if c.isalnum() or c in '- ')
        extraido, metodo = extraer_patron_placa(raw_limpio.strip(), tipo)
        detalle[nombre] = (raw_limpio.strip(), extraido, metodo)
        scores[nombre]  = score_ocr_resultado(extraido, tipo)

    mejor_var   = max(scores, key=lambda k: scores[k])
    texto_final = detalle[mejor_var][1]

    otsu_corr, _ = auto_invertir(b_otsu)
    t130 = bins.get('fijo_130', bins.get('fijo_110', b_otsu))
    t130_corr, _ = auto_invertir(t130)

    etapas = {
        'original'  : img_placa_rgb,
        'zona'      : img_base,
        'clahe'     : g_cla,
        'otsu_corr' : otsu_corr,
        't130_corr' : t130_corr,
    }
    return texto_final, detalle, scores, mejor_var, etapas

print('extraer_texto() lista ✅  (PSM adaptativo: PSM 7 normal, PSM 6 fallback)')


extraer_texto() lista ✅  (PSM adaptativo: PSM 7 normal, PSM 6 fallback)


In [9]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Procesamiento completo
# ═══════════════════════════════════════════════════════════════

imagenes   = cargar_imagenes(RUTA_PLACAS, prefijos=('placa',))
resultados = []

for ruta in imagenes:
    nombre = os.path.basename(ruta)
    print(f'\n{"="*62}')
    print(f'  {nombre}')
    print(f'{"="*62}')

    img_rgb, recortes, coords, intento_ok = detectar_placas(ruta)
    if img_rgb is None: continue

    es_fallback = (intento_ok == 0)   # ← PARCHE 4: saber si es fallback
    recorte     = recortes[0]
    tipo, info_tipo = identificar_tipo_placa(recorte)
    print(f'  Tipo: {tipo}  (EU={info_tipo["eu"]}%, CO={info_tipo["co"]}%)'
          f'  fallback={es_fallback}')

    texto, detalle, scores, mejor_var, etapas = extraer_texto(
        recorte, tipo=tipo, es_fallback=es_fallback)  # ← pasa el flag

    raw_text, texto_extraido, metodo = detalle[mejor_var]
    gt_texto, _ = GROUND_TRUTH.get(nombre, ('?','?'))
    resultado   = verificar_ocr(texto, gt_texto)

    print(f'  OCR raw      : [{raw_text[:60]}]')
    print(f'  OCR extraído : [{texto_extraido}]  via={mejor_var}({metodo})')
    print(f'  Esperado     : [{gt_texto}]  resultado: {resultado}')

    top3 = sorted(scores.items(), key=lambda x: -x[1])[:3]
    top3_str = '  |  '.join(f'{k}(s={v}):[{detalle[k][1]}]' for k,v in top3)
    print(f'  Top 3: {top3_str}')

    resultados.append({
        'imagen'     : nombre,
        'tipo'       : tipo,
        'texto'      : texto,
        'raw'        : raw_text,
        'metodo'     : metodo,
        'mejor'      : mejor_var,
        'intento'    : intento_ok,
        'es_fallback': es_fallback,
        'recorte'    : recorte,
        'etapas'     : etapas,
    })

print(f'\n\nProcesamiento completo: {len(resultados)} imagen(es) ✅')



  placa1.jpg
  Sin detecciones → fallback imagen completa
  Tipo: MX  (EU=0.0%, CO=0.0%)  fallback=True


NameError: name 'extraer_patron_placa' is not defined

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3g — verificar_ocr()  ← PARCHE 1: bug string vacío corregido
# ═══════════════════════════════════════════════════════════════
#
# BUG v6: normalizar('--') = '' y en Python '' in 'cualquier_string' = True
# Eso hacía que placa2, placa3, placa6, placa7 marcaran ✅ siendo vacías.
# El accuracy real era 0/7, no 57%.
# FIX: si ocr normalizado está vacío → siempre es ❌

def normalizar(texto):
    """Quita guiones, espacios y convierte a mayúsculas."""
    return ''.join(c for c in texto.upper() if c.isalnum())

def verificar_ocr(texto_ocr, ground_truth):
    """
    Verifica si el OCR es correcto con comparación normalizada.
    FIX: string vacío NUNCA es correcto.
    """
    if not ground_truth or ground_truth == '?':
        return ' — '
    ocr_n = normalizar(texto_ocr)
    gt_n  = normalizar(ground_truth)
    if not ocr_n:           # ← FIX: string vacío = siempre incorrecto
        return '❌'
    if ocr_n == gt_n or gt_n in ocr_n or ocr_n in gt_n:
        return '✅'
    return '❌'

print('verificar_ocr() lista ✅  (bug string vacío corregido)')
print('\nPrueba:')
tests = [
    ('NAA-77-41', 'NAA7741',  '✅'),
    ('LRV-59-77', 'LRV5977',  '✅'),
    ('--',        '5649JSN',  '❌ (era falso ✅ antes del fix)'),
    ('',          'NNR7087',  '❌ (era falso ✅ antes del fix)'),
    ('EDOMEX',    'NLU9603',  '❌'),
]
for ocr, gt, esperado in tests:
    r = verificar_ocr(ocr, gt)
    print(f'  [{ocr}] vs [{gt}] → {r}  (esperado: {esperado})')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Grilla visual con bounding boxes del OCR
# ═══════════════════════════════════════════════════════════════
#
# Col 1 → original + bbox del Haar (verde)
# Col 2 → zona crop (input al OCR)
# Col 3 → Otsu corregido
# Col 4 → Otsu corregido + bounding boxes del texto grande (amarillo)

n    = len(resultados)
fig, axes = plt.subplots(n, 4, figsize=(20, 3.2*n))
if n == 1: axes = [axes]

for fila, r in enumerate(resultados):
    et = r['etapas']
    gt_texto, _ = GROUND_TRUTH.get(r['imagen'], ('?','?'))

    # Col 1: original
    ax = axes[fila][0]
    ax.imshow(et['original'])
    ax.set_title(f"{r['imagen']} | {r['tipo']}\nOCR:[{r['texto']}] esp:[{gt_texto}]",
                 fontsize=7.5, pad=3)
    ax.axis('off')

    # Col 2: zona crop
    ax = axes[fila][1]
    ax.imshow(et['zona'])
    ax.set_title(f'Zona crop ({r["tipo"]})', fontsize=7.5, pad=3)
    ax.axis('off')

    # Col 3: Otsu corregido
    ax = axes[fila][2]
    ax.imshow(et['otsu_corr'], cmap='gray')
    ax.set_title('Otsu + auto-inv', fontsize=7.5, pad=3)
    ax.axis('off')

    # Col 4: Otsu + bounding boxes del texto grande
    ax = axes[fila][3]
    ax.imshow(et['otsu_corr'], cmap='gray')
    for (bx, by, bw, bh) in r['boxes']:
        rect = mpatches.Rectangle((bx, by), bw, bh,
                                   linewidth=2, edgecolor='yellow', facecolor='none')
        ax.add_patch(rect)
    n_boxes = len(r['boxes'])
    ax.set_title(f'Texto grande ({n_boxes} boxes)\n[{r["mejor"]}]', fontsize=7.5, pad=3)
    ax.axis('off')

plt.suptitle('Reconocimiento de Placas — v6 (text-height filtering)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output_placas_v6.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grilla guardada como output_placas_v6.png ✅')
